# Lab 07 — 10 Quality Scorecard



In [ ]:
from pathlib import Path
import sys
cwd=Path.cwd().resolve()
project_root=next((p for p in [cwd,*cwd.parents] if (p/'src'/'lab07').exists()),None)
if project_root and str(project_root/'src') not in sys.path: sys.path.insert(0,str(project_root/'src'))
if project_root and str(project_root/'tools') not in sys.path: sys.path.insert(0,str(project_root/'tools'))
dbutils.widgets.text('catalog','dbr_dev','01 Catalog'); dbutils.widgets.text('schema','parvinbadalov','02 Schema'); dbutils.widgets.text('volume_name','lab07_data_quality','03 Volume'); dbutils.widgets.text('run_id','manual','04 Run ID')
catalog=dbutils.widgets.get('catalog'); schema=dbutils.widgets.get('schema'); volume_name=dbutils.widgets.get('volume_name'); run_id=dbutils.widgets.get('run_id')
assert catalog=='dbr_dev' and schema=='parvinbadalov', f'Lab 07 requires dbr_dev.parvinbadalov, got {catalog}.{schema}'
volume_root=f'/Volumes/{catalog}/{schema}/{volume_name}'


In [ ]:
from pyspark.sql import functions as F
# Build scorecard from observable tables even when some independent checks only print evidence.
classified=spark.table(f'{catalog}.{schema}.business_license_classified'); total=classified.count(); quarantine=classified.filter("_dq_status='QUARANTINE'").count(); warnings=classified.filter("_dq_status='WARN'").count(); trusted=total-quarantine; score=round(trusted/total*100,2) if total else 0.0
scorecard=spark.createDataFrame([(run_id,'OVERALL',total,trusted,warnings,quarantine,float(score))],'run_id string,dimension string,checks long,passed long,warnings long,failed long,score_pct double'); scorecard.write.mode('overwrite').saveAsTable(f'{catalog}.{schema}.lab07_quality_scorecard'); display(scorecard)
